# Python II — Object-Oriented Programming
### Demo notebook — Lesson 1

In this notebook we build **one single example**, step by step: a small model of **animals in a shelter**.
We start from an empty class and, section after section, we add new features to it.

| # | Section | What we add |
|---|---------|-------------|
| 1 | Everything is an object | a look at objects we already know |
| 2 | Our first class | `class`, instances, `type`, `isinstance` |
| 3 | Initializing: `__init__` | instance attributes, `self` |
| 4 | Methods | behaviour that reads and changes the state |
| 5 | Class attributes | data shared by all instances |
| 6 | Inheritance | `Mammal(Animal)`, `Dog(Mammal)` |
| 7 | Overriding and `super()` | extending inherited methods |
| 8 | Special methods | `__repr__`, `__str__`, `__eq__`, `__lt__`, `__len__`, `__getitem__`, `__contains__`, `__add__` |
| 9 | Decorators | writing our own, then `@property`, `@classmethod`, `@staticmethod` |
| 10 | Abstract classes | `ABC` and `@abstractmethod` |

> **How to read this notebook:** every code cell is preceded by a markdown cell explaining what it does and why.
> Since we keep improving the *same* class, we will **re-define it several times**: the most recent definition is always the one in use.

---
## 1. Everything is an object

An **object** is any piece of data that has a **state** (the data it holds) and a **behaviour** (what it can do).
In Python *everything* is an object: numbers, strings, lists, even functions.

Every object has a **type**, which tells us what kind of object it is. We can inspect it with `type()`.

In [1]:
a = 1
s = "hello"
l = [1, 2, 3]

print(type(a))      # an int
print(type(s))      # a str
print(type(l))      # a list
print(type(print))  # even a function is an object!

<class 'int'>
<class 'str'>
<class 'list'>
<class 'builtin_function_or_method'>


The behaviour of an object is accessed with the **dot operator** (`.`).
For example, strings know how to turn themselves to uppercase, and lists know how to append new elements.
`int`, `str`, `list` are **classes** that are built into Python: today we learn how to write our own.

In [2]:
print(s.upper())   # behaviour of a str
l.append(4)        # behaviour of a list: it changes its own state
print(l)
print(print.__doc__[:60], "...")  # a function also has state (its documentation)

HELLO
[1, 2, 3, 4]
Prints the values to a stream, or to sys.stdout by default.
 ...


---
## 2. Our first class

A **class** is a *template* for creating objects. We define it with the keyword `class`, followed by its name
(by convention in `CamelCase`).

Here is the smallest possible class: it contains nothing. `pass` is just a placeholder meaning "do nothing",
needed because Python does not allow an empty block.

In [3]:
class Animal:
    pass

The class itself is only the template. To get an actual object we **instantiate** the class by *calling* it,
as if it were a function. The resulting object is called an **instance** of the class.

We can create as many instances as we want: each one is a **distinct object**.

In [4]:
a1 = Animal()
a2 = Animal()

print(a1)
print(a2)
print(type(a1))

<class '__main__.Animal'>


- `type(a1)` tells us that `a1` is an `Animal`.
- `isinstance(obj, Class)` checks whether an object is an instance of a class.
- `a1` and `a2` are built from the same template but are **two different objects** (`is` checks identity).

In [5]:
print(isinstance(a1, Animal))   # True
print(isinstance(5, Animal))    # False
print(a1 is a2)                 # False: two separate objects

True
False
False


---
## 3. Initializing: `__init__` and instance attributes

An empty animal is not very useful: we want each animal to have a **name** and an **age**.

`__init__` is a **special method** (double underscores on both sides, also called *dunder* method) that Python
runs automatically **every time a new instance is created**. Its job is to set up the initial state of the object.

About `self`:
- the first parameter of every method is `self`, and it refers to **the instance being created/used**;
- `self.name = name` creates an **instance attribute**: a variable that belongs to *that specific* instance;
- when we create the object we **do not pass `self`**: Python does it for us.

In [6]:
class Animal:                          # version 2
    def __init__(self, name, age):
        self.name = name               # instance attribute
        self.age = age                 # instance attribute

Now the arguments of `__init__` (except `self`) are passed when instantiating.
Each instance stores **its own** values, which we read with the dot operator.

In [7]:
rex = Animal("Rex", 3)
luna = Animal("Luna", 5)

print(rex.name, rex.age)
print(luna.name, luna.age)

Rex 3
Luna 5


Instance attributes can be modified after creation, and changing one instance does **not** affect the others.

In [8]:
rex.age = 4
print(rex.age)    # 4
print(luna.age)   # still 5

4
5


Since `__init__` now requires `name` and `age`, forgetting them raises an error.
(We wrap the call in `try/except` so the notebook keeps running and we can read the message.)

In [9]:
try:
    nameless = Animal()
except TypeError as err:
    print("TypeError:", err)

TypeError: Animal.__init__() missing 2 required positional arguments: 'name' and 'age'


---
## 4. Methods: adding behaviour

A **method** is simply a function defined inside a class. It always receives the instance as first argument (`self`),
so it can **read** and **modify** the state of that instance.

We add two methods:
- `info()` prints a description of the animal (it *reads* the state);
- `birthday()` increases the age by one (it *changes* the state).

In [10]:
class Animal:                          # version 3
    def __init__(self, name, age):
        self.name = name
        self.age = age

    def info(self):
        print(f"I am {self.name} and I am {self.age} years old.")

    def birthday(self):
        self.age += 1
        print(f"Happy birthday {self.name}! Now {self.age} years old.")

Note that we recreate `rex` and `luna`: the old objects were built with the *old* version of the class,
which did not have these methods.

In [11]:
rex = Animal("Rex", 3)
luna = Animal("Luna", 5)

rex.info()
rex.birthday()
rex.info()
luna.info()     # luna was not affected

I am Rex and I am 3 years old.
Happy birthday Rex! Now 4 years old.
I am Rex and I am 4 years old.
I am Luna and I am 5 years old.


---
## 5. Class attributes

Sometimes data is the same for **all** instances. In that case we define it directly in the class body,
outside any method: this is a **class attribute**, shared by every instance.

We add:
- `kingdom = "Animalia"`: true for every animal;
- `count = 0`: a counter of how many animals have been created. In `__init__` we update it through the
  **class name** (`Animal.count += 1`), because we want to change the shared value.

In [13]:
class Animal:                          # version 4
    kingdom = "Animalia"               # class attribute
    count = 0                          # class attribute

    def __init__(self, name, age):
        self.name = name               # instance attribute
        self.age = age                 # instance attribute
        Animal.count += 1

    def info(self):
        print(f"I am {self.name} and I am {self.age} years old.")

    def birthday(self):
        self.age += 1
        print(f"Happy birthday {self.name}! Now {self.age} years old.")

Class attributes can be read both from the **class** and from any **instance**.
Instance attributes, instead, only exist on instances: asking the class for `age` fails.

In [14]:
rex = Animal("Rex", 3)
luna = Animal("Luna", 5)

print(Animal.kingdom, rex.kingdom, luna.kingdom)
print("Animals created:", Animal.count)

try:
    Animal.age
except AttributeError as err:
    print("AttributeError:", err)

Animalia Animalia Animalia
Animals created: 2
AttributeError: type object 'Animal' has no attribute 'age'


⚠️ **A common pitfall: shadowing.**

If we *assign* a class attribute through an instance (`rex.kingdom = ...`), Python does **not** change the class:
it creates a **new instance attribute** on `rex` with the same name, which hides the class one.
The other instances are not affected. This is why we wrote `Animal.count += 1` and not `self.count += 1`.

`vars(obj)` shows the instance attributes stored on an object.

In [15]:
rex.kingdom = "Robots"

print(rex.kingdom)      # Robots  (instance attribute)
print(luna.kingdom)     # Animalia (class attribute)
print(Animal.kingdom)   # Animalia
print(vars(rex))        # 'kingdom' is now stored on rex itself
print(vars(luna))

Robots
Animalia
Animalia
{'name': 'Rex', 'age': 3, 'kingdom': 'Robots'}
{'name': 'Luna', 'age': 5}


---
## 6. Inheritance

Classes can **inherit** attributes and methods from other classes, in order to **extend** them.
The syntax is `class Child(Parent):`.

We build a small hierarchy, as in the slides:
- `Animal` → the *grandparent*, the general concept;
- `Mammal(Animal)` → the *parent*: an animal, plus a new class attribute `is_hairy`;
- `Dog(Mammal)` → the *child*: a mammal, plus a new method `woof()`.

Neither `Mammal` nor `Dog` defines `__init__`, `info` or `birthday`: they inherit them from `Animal`.

In [16]:
class Mammal(Animal):
    is_hairy = True

class Dog(Mammal):
    def woof(self):
        print(f"{self.name} says: Woof!")

In [17]:
rex = Dog("Rex", 3)       # __init__ comes from Animal

rex.info()                # inherited from Animal
rex.woof()                # defined in Dog
print(rex.is_hairy)       # inherited from Mammal
print(rex.kingdom)        # inherited from Animal

I am Rex and I am 3 years old.
Rex says: Woof!
True
Animalia


A `Dog` **is** also a `Mammal` and an `Animal`, so `isinstance` returns `True` for all of them.
The reverse is not true: a generic `Animal` is not a `Dog`.

`issubclass` checks relationships between classes, and the **MRO** (*Method Resolution Order*) shows the order
in which Python searches for an attribute: first in `Dog`, then `Mammal`, then `Animal`, and finally `object`,
the class every Python class implicitly inherits from.

In [18]:
generic = Animal("Generic", 1)

print(isinstance(rex, Dog), isinstance(rex, Mammal), isinstance(rex, Animal))
print(isinstance(generic, Dog))
print(issubclass(Dog, Animal))
print(Dog.__mro__)

True True True
False
True
(<class '__main__.Dog'>, <class '__main__.Mammal'>, <class '__main__.Animal'>, <class 'object'>)


A generic `Animal` does not have the `woof` method: inheritance goes from parent to child, never the other way.

In [19]:
try:
    generic.woof()
except AttributeError as err:
    print("AttributeError:", err)

AttributeError: 'Animal' object has no attribute 'woof'


---
## 7. Overriding and `super()`

A child class can **override** an inherited method by defining a method with the **same name**.
Usually we do not want to throw away the parent's code but to **extend** it. We can call the original method in two ways:
- `super().method(...)`: "the same method, as defined in my parent" (the recommended way);
- `ParentClass.method(self, ...)`: the explicit class name (works, but must be updated if the hierarchy changes).

We give `Dog` a new attribute, `breed`:
- in `__init__` we let `Animal.__init__` set `name` and `age` (via `super()`), then add `breed`;
- in `info` we print the generic information first, then the dog-specific part.

In [24]:
class Dog(Mammal):                                     # version 2
    def __init__(self, name, age, breed):
        super().__init__(name, age)    # runs Animal.__init__: sets name, age, updates count
        self.breed = breed             # extra attribute, only for dogs

    def info(self):
        super().info()                 # equivalent here to: Animal.info(self)
        print(f"I am a {self.breed}.")

    def woof(self):
        print(f"{self.name} says: Woof!")

In [25]:
rex = Dog("Rex", 3, "Labrador")
rex.info()          # overridden version
rex.birthday()      # not overridden: Animal's version is used

I am Rex and I am 3 years old.
I am a Labrador.
Happy birthday Rex! Now 4 years old.


💡 Try removing the `super().__init__(name, age)` line and run `rex.info()` again: you will get an
`AttributeError`, because nobody sets `self.name` anymore. When you override `__init__`, remember to call the parent's one.

---
## 8. Special (dunder) methods

We already met `__init__`. Python has many other **special methods** that are called *implicitly* by built-in
operations. Implementing them makes our objects behave like native Python objects:

| You write | Python calls |
|-----------|--------------|
| `repr(x)`, or `x` alone in a notebook cell | `x.__repr__()` |
| `print(x)`, `str(x)` | `x.__str__()` |
| `x == y` | `x.__eq__(y)` |
| `x < y`, `sorted(...)` | `x.__lt__(y)` |
| `len(x)` | `x.__len__()` |
| `x[i]` | `x.__getitem__(i)` |
| `item in x` | `x.__contains__(item)` |
| `x + y` | `x.__add__(y)` |

### 8.1 The problem
Look at what happens now when we print an animal: we only get its type and memory address, not very informative.

In [26]:
print(rex)
rex

### 8.2 `__repr__` and `__str__`

- `__repr__` returns an **unambiguous** representation, aimed at developers. Good practice: make it look like the
  code that would recreate the object. Notebooks and the debugger use it.
- `__str__` returns a **readable** representation, aimed at users. `print()` uses it.
  If `__str__` is missing, Python falls back on `__repr__`.

`type(self).__name__` gives the name of the actual class (`Animal`, `Dog`, ...), so the same `__repr__`
works correctly for subclasses.

### 8.3 `__eq__` and `__lt__`

- By default `==` behaves like `is`: two objects are equal only if they are *the same object*.
  With `__eq__` we decide that two animals are equal if they have the same class, name and age.
  If the other object is not an `Animal` we return `NotImplemented`, which tells Python "I don't know how to compare these".
- `__lt__` (*less than*) defines `<`. We order animals by age. Once `<` is defined, `sorted()`, `min()`, `max()` work too.

Since we are re-defining `Animal`, we also need to re-define `Mammal` and `Dog`:
⚠️ a subclass is bound to the parent class object that existed **when it was defined**, not to its name.

In [27]:
class Animal:                                          # version 5
    kingdom = "Animalia"
    count = 0

    def __init__(self, name, age):
        self.name = name
        self.age = age
        Animal.count += 1

    def info(self):
        print(f"I am {self.name} and I am {self.age} years old.")

    def birthday(self):
        self.age += 1
        print(f"Happy birthday {self.name}! Now {self.age} years old.")

    # --- special methods ---
    def __repr__(self):
        return f"{type(self).__name__}({self.name!r}, {self.age!r})"

    def __str__(self):
        return f"{self.name} ({self.age} y.o.)"

    def __eq__(self, other):
        if not isinstance(other, Animal):
            return NotImplemented
        return (type(self), self.name, self.age) == (type(other), other.name, other.age)

    def __lt__(self, other):
        if not isinstance(other, Animal):
            return NotImplemented
        return self.age < other.age


class Mammal(Animal):
    is_hairy = True


class Dog(Mammal):
    def __init__(self, name, age, breed="unknown"):
        super().__init__(name, age)
        self.breed = breed

    def __repr__(self):                                # overriding a special method!
        return f"Dog({self.name!r}, {self.age!r}, {self.breed!r})"

    def info(self):
        super().info()
        print(f"I am a {self.breed}.")

    def woof(self):
        print(f"{self.name} says: Woof!")

`repr` vs `str` in action:

In [28]:
rex = Dog("Rex", 3, "Labrador")
luna = Mammal("Luna", 5)

print(rex)          # __str__ (inherited from Animal)
print(repr(rex))    # __repr__ (overridden in Dog)
print(luna)         # __str__
luna                # last expression of a cell -> __repr__

Rex (3 y.o.)
Dog('Rex', 3, 'Labrador')
Luna (5 y.o.)


Mammal('Luna', 5)

Equality and ordering:

In [29]:
a = Animal("Milo", 2)
b = Animal("Milo", 2)
c = Animal("Nala", 7)

print(a == b)       # True: same class, name and age (thanks to __eq__)
print(a is b)       # False: still two distinct objects
print(a == c)       # False
print(a < c)        # True: 2 < 7  (thanks to __lt__)
print(c > a)        # True: Python uses a.__lt__(c) as a fallback for >

animals = [c, rex, luna, a]
print(sorted(animals))                # sorted by age
print("Youngest:", min(animals))

True
False
False
True
True
[Animal('Milo', 2), Dog('Rex', 3, 'Labrador'), Mammal('Luna', 5), Animal('Nala', 7)]
Youngest: Milo (2 y.o.)


### 8.4 A container class: `Shelter`

To show container-like special methods, we build a second class that **holds** our animals:
a `Shelter` that stores a list of animals.

- `__len__` → `len(shelter)` returns the number of animals;
- `__getitem__` → `shelter[0]` and `shelter[1:3]` work. As a bonus, Python can also **loop** over an object that has
  `__getitem__`, so `for animal in shelter` works for free;
- `__contains__` → `animal in shelter`. It uses `==`, so it relies on the `__eq__` we wrote above;
- `__add__` → `shelter1 + shelter2` returns a **new** shelter with the animals of both.

In [30]:
class Shelter:
    def __init__(self, name, animals=None):
        self.name = name
        # avoid using a mutable default like animals=[] : it would be shared by all shelters!
        self.animals = list(animals) if animals is not None else []

    def admit(self, animal):
        self.animals.append(animal)

    def __repr__(self):
        return f"Shelter({self.name!r}, {self.animals!r})"

    def __len__(self):
        return len(self.animals)

    def __getitem__(self, index):
        return self.animals[index]

    def __contains__(self, animal):
        return animal in self.animals

    def __add__(self, other):
        if not isinstance(other, Shelter):
            return NotImplemented
        return Shelter(f"{self.name} & {other.name}", self.animals + other.animals)

In [31]:
north = Shelter("North", [rex, luna])
south = Shelter("South")
south.admit(Animal("Milo", 2))
south.admit(Dog("Fido", 8, "Beagle"))

print(len(north), len(south))           # __len__
print(north[0])                          # __getitem__ with an index
print(Animal("Milo", 2) in south)        # __contains__ (+ __eq__): True, even if it's a new object

merged = north + south                   # __add__
print(merged)
print(merged[1:3])                       # slicing works as well

for animal in merged:                    # iteration, thanks to __getitem__
    print(" -", animal)

print("Oldest in the merged shelter:", max(merged))   # max uses __lt__

2 2
Rex (3 y.o.)
True
Shelter('North & South', [Dog('Rex', 3, 'Labrador'), Mammal('Luna', 5), Animal('Milo', 2), Dog('Fido', 8, 'Beagle')])
[Mammal('Luna', 5), Animal('Milo', 2)]
 - Rex (3 y.o.)
 - Luna (5 y.o.)
 - Milo (2 y.o.)
 - Fido (8 y.o.)
Oldest in the merged shelter: Fido (8 y.o.)


---
## 9. Decorators

### 9.1 Functions are objects
Remember: in Python functions are objects too. This means a function can be **passed to** another function and
**returned by** another function, just like an integer or a list.

In [33]:
def shout(text):
    return text.upper() + "!"

yell = shout            # no parentheses: we are not calling it, just giving it another name
print(yell("hello"))

def apply_twice(func, value):
    return func(func(value))

print(apply_twice(shout, "hey"))

HELLO!
HEY!!


### 9.2 Writing our own decorator

A **decorator** is a function that takes a function and returns a **new function** that wraps the original one,
adding some behaviour *before* and/or *after* it, without modifying its code.

Our decorator `log_call` prints which function is called and with which arguments, then calls the original function:
- `func` is the original function;
- `wrapper` is the new function; `*args, **kwargs` let it accept **any** arguments and pass them on unchanged;
- `@functools.wraps(func)` copies name and docstring of `func` onto `wrapper` (good practice, otherwise
  every decorated function would be called "wrapper").

In [34]:
import functools

def log_call(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print(f"[LOG] calling {func.__name__} with args={args} kwargs={kwargs}")
        result = func(*args, **kwargs)
        print(f"[LOG] {func.__name__} returned {result!r}")
        return result
    return wrapper

The `@` syntax is just a shortcut. Writing

```python
@log_call
def add(x, y): ...
```
is **exactly the same** as writing `add = log_call(add)` after the definition.

In [35]:
@log_call
def add(x, y):
    return x + y

add(2, 3)

[LOG] calling add with args=(2, 3) kwargs={}
[LOG] add returned 5


5

Decorators work on methods too, since methods are functions defined inside a class.
Here we decorate `birthday` in a subclass. Note that `args` also contains `self` (the animal), printed with its `__repr__`.

In [36]:
class LoggedDog(Dog):
    @log_call
    def birthday(self):
        super().birthday()

fido = LoggedDog("Fido", 8, "Beagle")
fido.birthday()

[LOG] calling birthday with args=(Dog('Fido', 8, 'Beagle'),) kwargs={}
Happy birthday Fido! Now 9 years old.
[LOG] birthday returned None


### OPTIONAL: 9.3 Built-in decorators for classes

Python comes with three very useful decorators for classes:

- **`@property`** turns a method into an attribute that is read *without parentheses*. Together with its
  `@<name>.setter`, it lets us run code (e.g. a **validation**) every time the attribute is read or assigned.
  The actual value is stored in `self._age`: the leading underscore is a *convention* meaning "internal, don't touch it directly".
- **`@staticmethod`** defines a method that receives **neither** `self` **nor** the class: it is a plain function
  that lives inside the class because it is logically related to it. Here: `is_valid_age(value)`.
- **`@classmethod`** defines a method that receives the **class** (`cls`) instead of the instance. It is typically used
  to write **alternative constructors**. Here: `from_string("Rex,3")`. Since it calls `cls(...)` and not `Animal(...)`,
  it automatically creates the right type when called on a subclass (`Dog.from_string(...)` creates a `Dog`).

Note that in `__init__` we write `self.age = age`: this already goes through the setter, so the validation is applied
at creation time as well.

In [ ]:
class Animal:                                          # version 6
    kingdom = "Animalia"
    count = 0

    def __init__(self, name, age):
        self.name = name
        self.age = age               # goes through the property setter below
        Animal.count += 1

    # --- property ---
    @property
    def age(self):
        return self._age

    @age.setter
    def age(self, value):
        if not Animal.is_valid_age(value):
            raise ValueError(f"Invalid age: {value!r}")
        self._age = value

    # --- static method ---
    @staticmethod
    def is_valid_age(value):
        return isinstance(value, int) and value >= 0

    # --- regular methods ---
    def info(self):
        print(f"I am {self.name} and I am {self.age} years old.")

    def birthday(self):
        self.age += 1
        print(f"Happy birthday {self.name}! Now {self.age} years old.")

    # --- special methods ---
    def __repr__(self):
        return f"{type(self).__name__}({self.name!r}, {self.age!r})"

    def __str__(self):
        return f"{self.name} ({self.age} y.o.)"

    def __eq__(self, other):
        if not isinstance(other, Animal):
            return NotImplemented
        return (type(self), self.name, self.age) == (type(other), other.name, other.age)

    def __lt__(self, other):
        if not isinstance(other, Animal):
            return NotImplemented
        return self.age < other.age


class Mammal(Animal):
    is_hairy = True


class Dog(Mammal):
    def __init__(self, name, age, breed="unknown"):
        super().__init__(name, age)
        self.breed = breed

    def __repr__(self):
        return f"Dog({self.name!r}, {self.age!r}, {self.breed!r})"

    def info(self):
        super().info()
        print(f"I am a {self.breed}.")

    def woof(self):
        print(f"{self.name} says: Woof!")

**`@property`**: `age` is read and assigned like a normal attribute, but invalid values are now rejected,
both when creating the object and when modifying it.

In [38]:
rex = Dog("Rex", 3, "Labrador")
print(rex.age)          # no parentheses: the property getter runs
rex.age = 4             # the setter runs and validates
print(rex.age)

for bad in (-2, "three"):
    try:
        rex.age = bad
    except ValueError as err:
        print("ValueError:", err)

try:
    Dog("Ghost", -1)
except ValueError as err:
    print("ValueError at creation:", err)

3
4
ValueError: Invalid age: -2
ValueError: Invalid age: 'three'
ValueError at creation: Invalid age: -1


**`@staticmethod`**: it can be called both on the class and on an instance, and it does not need either of them.

In [ ]:
print(Animal.is_valid_age(5))
print(Animal.is_valid_age(-5))
print(rex.is_valid_age(2.5))

---
## OPTIONAL: 10. Abstract classes

Look at our hierarchy: does it make sense to create a generic `Animal`? In the real world there is no "generic animal",
only dogs, cats, etc. `Animal` exists to define **what every animal must have and be able to do**.

An **abstract class** is a class that:
- **cannot be instantiated** directly;
- declares one or more **abstract methods**: methods that every concrete subclass is **obliged** to implement.

In Python we get this from the `abc` module (*Abstract Base Classes*): we inherit from `ABC` and mark the
required methods with `@abstractmethod`.

Here every animal must know how to `speak()`. To keep the example short, this final version contains only the essential
parts (the full version is in the summary at the end).

In [ ]:
from abc import ABC, abstractmethod

class Animal(ABC):                                     # version 7: abstract
    def __init__(self, name, age):
        self.name = name
        self.age = age

    @abstractmethod
    def speak(self):
        # No implementation here: subclasses MUST provide one.
        ...

    def introduce(self):
        # A concrete method that relies on the abstract one.
        print(f"I am {self.name}, {self.age} y.o., and I say: {self.speak()}")

    def __repr__(self):
        return f"{type(self).__name__}({self.name!r}, {self.age!r})"

Trying to instantiate the abstract class fails:

In [ ]:
try:
    generic = Animal("Generic", 1)
except TypeError as err:
    print("TypeError:", err)

A subclass that **forgets** to implement `speak` is still abstract, so it cannot be instantiated either.
Python catches the mistake **when the object is created**, not later when `speak()` would be called.

In [ ]:
class Fish(Animal):
    pass                       # no speak() !

try:
    nemo = Fish("Nemo", 1)
except TypeError as err:
    print("TypeError:", err)

Concrete subclasses implement `speak`, and they can be instantiated normally.

This also shows **polymorphism**: the same call, `animal.introduce()`, produces a different result depending on the
actual type of the object. The loop does not need to know which kind of animal it is dealing with.

In [ ]:
class Dog(Animal):
    def speak(self):
        return "Woof!"

class Cat(Animal):
    def speak(self):
        return "Meow!"

class Cow(Animal):
    def speak(self):
        return "Moo!"

farm = [Dog("Rex", 3), Cat("Luna", 5), Cow("Bella", 4)]

for animal in farm:
    animal.introduce()

---
## Summary: the complete example

Here is everything we built, put together in a single cell: an abstract `Animal` with properties, static and
class methods and special methods; concrete subclasses using inheritance and overriding; a `Shelter` container;
and a custom decorator.

In [ ]:
import functools
from abc import ABC, abstractmethod


def log_call(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print(f"[LOG] {func.__name__}{args[1:]}")      # skip self in the log
        return func(*args, **kwargs)
    return wrapper


class Animal(ABC):
    kingdom = "Animalia"                 # class attribute
    count = 0

    def __init__(self, name, age):
        self.name = name                 # instance attribute
        self.age = age                   # validated by the property
        Animal.count += 1

    @property
    def age(self):
        return self._age

    @age.setter
    def age(self, value):
        if not Animal.is_valid_age(value):
            raise ValueError(f"Invalid age: {value!r}")
        self._age = value

    @staticmethod
    def is_valid_age(value):
        return isinstance(value, int) and value >= 0

    def introduce(self):
        print(f"I am {self}, and I say: {self.speak()}")

    @log_call
    def birthday(self):
        self.age += 1

    def __repr__(self):
        return f"{type(self).__name__}({self.name!r}, {self.age!r})"

    def __str__(self):
        return f"{self.name} ({self.age} y.o.)"

    def __eq__(self, other):
        if not isinstance(other, Animal):
            return NotImplemented
        return (type(self), self.name, self.age) == (type(other), other.name, other.age)

    def __lt__(self, other):
        if not isinstance(other, Animal):
            return NotImplemented
        return self.age < other.age


class Mammal(Animal):
    is_hairy = True


class Dog(Mammal):
    def __init__(self, name, age, breed="unknown"):
        super().__init__(name, age)
        self.breed = breed

    def speak(self):
        return "Woof!"

    def introduce(self):                 # overriding + super()
        super().introduce()
        print(f"   ...and I am a {self.breed}.")


class Cat(Mammal):
    def speak(self):
        return "Meow!"


class Shelter:
    def __init__(self, name, animals=None):
        self.name = name
        self.animals = list(animals) if animals is not None else []

    def admit(self, animal):
        self.animals.append(animal)

    def __repr__(self):
        return f"Shelter({self.name!r}, {self.animals!r})"

    def __len__(self):
        return len(self.animals)

    def __getitem__(self, index):
        return self.animals[index]

    def __contains__(self, animal):
        return animal in self.animals

    def __add__(self, other):
        if not isinstance(other, Shelter):
            return NotImplemented
        return Shelter(f"{self.name} & {other.name}", self.animals + other.animals)

In [ ]:
north = Shelter("North", [Dog("Rex", 3, "Labrador"), Cat.from_string("Luna, 5")])
south = Shelter("South", [Cat("Tom", 1), Dog("Fido", 8, "Beagle")])

everyone = north + south
print(everyone, "\n")

for animal in sorted(everyone):
    animal.introduce()

print()
everyone[0].birthday()
print(everyone[0])
print("Is Tom (1 y.o.) here?", Cat("Tom", 1) in everyone)

---
## Exercises

1. Add a `Bird(Animal)` class that implements `speak()` and has an extra attribute `can_fly` (default `True`).
2. Give `Shelter` an `adopt(name)` method that removes and returns the animal with that name
   (raise a `KeyError` if it does not exist).
3. Implement `__hash__` in `Animal` so that animals can be put in a `set`. Which attributes should you use? What happens
   if an animal has a birthday after being inserted in the set?
4. Write a decorator `count_calls` that counts how many times a function has been called
   (hint: you can store the counter as an attribute of `wrapper`).
5. Make `Shelter` iterable in a more explicit way by implementing `__iter__`.
   Does the `for` loop still work if you remove `__getitem__`? And indexing?